# Step-wise Confidence Estimation — End-to-End Demo (MoreHopQA, 10 cases)

This notebook walks the **entire pipeline** on 10 MoreHopQA cases:

1. **Generate** reasoning traces  →  2. **Transform** into graphs  →
3. **BERT** embeddings  →  4. **Label** step correctness (LLM judge)  →
5. **Run methods** (NIBS, GIBS, P(True), white-box)  →  6. **Evaluate** (AUROC / AUPRC).

Everything uses **relative paths**, so it runs straight after cloning the repo.
All intermediate artifacts are written under `output/morehopqa/deepseek/`.

> **GPU / API gating.** Some stages need a CUDA GPU (vLLM, DeBERTa, the baseline
> LLMs) or an OpenAI key. Those cells detect what's available: if the requirement
> is missing they print the exact command to run on a proper machine and continue.
> CPU-only stages (transform, BERT-on-CPU, final metrics) always run for real, and
> the **final evaluation falls back to bundled paper results** (`data/MorehopQA/Deepseek/`)
> so you always see real numbers.

## 0. Setup — work from the repo root and pick dataset/model

In [ ]:
import os, sys, subprocess, json, pickle, shutil

# Make sure we run from the repository root (this notebook lives in notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
REPO_ROOT = os.getcwd()
print("Repo root:", REPO_ROOT)

# Let us import the shared path/config module
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "dataset"))
import dataset_config as dc

DATASET      = "morehopqa"
MODEL        = "deepseek"                             # llama | deepseek | phi4
OUTPUT_ROOT  = "output"
SOURCE       = "data/source/morehopqa_sample.json"   # 10 cases
NUM_RESPONSES = 5                                   # samples per question (small for the demo)

# The baseline scripts (p_true / white_box) use a slightly different model key.
WB_MODEL = dc.WHITEBOX_MODEL_KEY[MODEL]              # deepseek -> "deepseek", llama -> "llama3"

# Canonical artifact paths (all derived in ONE place: dataset_config.py)
P = dict(
    responses   = dc.responses_path(OUTPUT_ROOT, DATASET, MODEL),
    transformed = dc.transformed_path(OUTPUT_ROOT, DATASET, MODEL),
    bert        = dc.bert_path(OUTPUT_ROOT, DATASET, MODEL),
    labels      = dc.evaluation_path(OUTPUT_ROOT, DATASET, MODEL),
    nibs        = dc.nibs_path(OUTPUT_ROOT, DATASET, MODEL),
    gibs        = dc.gibs_path(OUTPUT_ROOT, DATASET, MODEL),
    gib_ckpt    = dc.gib_checkpoint_dir(OUTPUT_ROOT, DATASET, MODEL),
    ptrue       = dc.ptrue_path(OUTPUT_ROOT, DATASET, MODEL),
    white_box   = dc.white_box_path(OUTPUT_ROOT, DATASET, MODEL),
)
for k, v in P.items():
    print(f"{k:11s} -> {v}")

# Bundled pre-computed paper results used as a no-GPU/no-key fallback in Stage 6.
BUNDLED = dict(
    labels    = "data/MorehopQA/Deepseek/evaluation.json",
    gibs      = "data/MorehopQA/Deepseek/GIBS_results.json",
    nibs      = "data/MorehopQA/Deepseek/NIBS_results.json",
    ptrue     = "data/MorehopQA/Deepseek/ptrue_deepseek_morehopqa.pkl",
    white_box = "data/MorehopQA/Deepseek/white_box_baseline_deepseek_r1.json",
)

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:
    pass
HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("\nCUDA available:", HAS_GPU, "| OPENAI_API_KEY set:", HAS_OPENAI_KEY)

def run(cmd):
    """Run a shell command from the repo root and stream a trimmed output."""
    print("$", " ".join(cmd))
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = res.stdout.decode("utf-8", "replace")
    print("\n".join(out.strip().splitlines()[-20:]))
    return res.returncode

In [ ]:
# Inspect the 10 demo cases
sample = json.load(open(SOURCE))
print(f"{len(sample)} cases. First one:")
print("  _id     :", sample[0]["_id"])
print("  question:", sample[0]["question"])
print("  answer  :", sample[0]["answer"])

## 1. Generate reasoning traces  *(GPU + vLLM)*

Real command:

```bash
python src/dataset/generate_responds_vllm.py --dataset morehopqa --model deepseek \
    --dataset-path data/source/morehopqa_sample.json --num-responses 5 --max-questions 10
```

If no GPU is available we instead write a small **illustrative** responses file so
the CPU stages below can run. It is clearly marked — replace it with real
generation output for actual experiments.

In [ ]:
os.makedirs(os.path.dirname(P["responses"]), exist_ok=True)

if HAS_GPU:
    run([sys.executable, "src/dataset/generate_responds_vllm.py",
         "--dataset", DATASET, "--model", MODEL,
         "--dataset-path", SOURCE,
         "--num-responses", str(NUM_RESPONSES), "--max-questions", "10"])
else:
    print("No GPU detected -> writing an ILLUSTRATIVE placeholder responses file.")
    print("(For real results, run the command above on a CUDA machine with vLLM.)\n")
    # Build a minimal but valid ReasoningGraph per case so the parser has something
    # to chew on. These are NOT real model samples — they are illustrative only.
    def fake_graph(q, ans, correct=True):
        a = ans if correct else "WRONG"
        return (f"ReasoningGraph(nodes=["
                f"ReasoningNode(id=1, description='Read the question', output='parsed', depends_on=[]), "
                f"ReasoningNode(id=2, description='Derive the answer', output='{a}', depends_on=[1])"
                f"], final_answer='{a}')")
    results = {}
    for d in sample:
        responses, rids, logps = [], [], []
        for i in range(NUM_RESPONSES):
            responses.append(fake_graph(d["question"], d["answer"], correct=(i % 2 == 0)))
            rids.append(i); logps.append(None)
        results[d["_id"]] = {
            "id": d["_id"], "question": d["question"], "context": d.get("context", ""),
            "answer": d["answer"], "response_ids": rids, "responses": responses,
            "token_ids": [[]] * NUM_RESPONSES, "logprobs": logps,
        }
    pickle.dump(results, open(P["responses"], "wb"))
    print("Wrote", P["responses"], "with", len(results), "questions x", NUM_RESPONSES, "responses")

## 2. Transform traces into structured graphs  *(CPU — runs for real)*

In [ ]:
run([sys.executable, "src/dataset/tranformed_graph.py", "--dataset", DATASET, "--model", MODEL])

t = pickle.load(open(P["transformed"], "rb"))
pid = next(iter(t)); gid = next(iter(t[pid]))
print("\nExample transformed graph:")
print("  structure_retrieve:", t[pid][gid]["structure_retrieve"])
print("  string_q_a        :", t[pid][gid]["string_q_a"])
print("  response_str      :", t[pid][gid]["response_str"])

## 3. BERT embeddings per step  *(GPU or CPU)*

Uses `bert-base-uncased`. We pass `--device cpu` automatically when there is no GPU
(downloads the ~420 MB model the first time).

In [ ]:
dev = "cuda" if HAS_GPU else "cpu"
run([sys.executable, "src/dataset/bert_graph.py",
     "--dataset", DATASET, "--model", MODEL, "--device", dev])

b = pickle.load(open(P["bert"], "rb"))
pid = next(iter(b)); gid = next(iter(b[pid]))
emb = b[pid][gid]["embeddings"]
print("\nEmbedding keys for one graph:", list(emb.keys()))
print("One embedding shape:", next(iter(emb.values())).shape)

## 4. Label step correctness with an LLM judge  *(OpenAI key)*

Real command:

```bash
export OPENAI_API_KEY=sk-...
python src/evaluation/gpt_eval.py --dataset morehopqa --model deepseek \
    --source data/source/morehopqa_sample.json
```

Without a key this runs a **dry run**: `evaluation.json` gets the correct schema
but every label is `-1` (unknown). For meaningful metrics, set a real key here or
reuse a pre-computed `evaluation.json`.

In [ ]:
cmd = [sys.executable, "src/evaluation/gpt_eval.py",
       "--dataset", DATASET, "--model", MODEL, "--source", SOURCE]
if not HAS_OPENAI_KEY:
    print("No OPENAI_API_KEY -> dry run (labels will be -1).\n")
run(cmd)

labels = json.load(open(P["labels"]))
pid = next(iter(labels)); gid = next(iter(labels[pid]))
print("\nExample label record:")
print(json.dumps(labels[pid][gid]["evaluations"][:2], indent=2))

## 5. Run the confidence-estimation methods  *(GPU)*

NIBS (DeBERTa-MNLI), GIBS (train + infer) and the two baselines all need a GPU and
load large models, so here we only **execute them when a GPU is present**;
otherwise the exact commands are printed for you to run on a proper machine.

In [ ]:
def gib_latest_ckpt():
    d = P["gib_ckpt"]
    if not os.path.isdir(d):
        return None
    cands = [f for f in os.listdir(d) if f.startswith("gib_model_") and f.endswith(".pth")]
    return os.path.join(d, sorted(cands)[-1]) if cands else None

nibs_cmd = [sys.executable, "src/NIBS/similarity_based_baseline.py",
            "--label-file", P["labels"], "--embedding-file", P["bert"],
            "--output-file", P["nibs"], "--device", "cuda"]

gib_train_cmd = [sys.executable, "src/GIBS/GIB_MCS_wo_mcs_feature.py",
                 "--embedding-file", P["bert"], "--label-file", P["labels"],
                 "--save-dir", P["gib_ckpt"], "--epochs", "200"]

ptrue_cmd = [sys.executable, "src/baseline/p_true.py",
             "--model", WB_MODEL, "--dataset", "morehopqa",
             "--data_path", P["transformed"], "--save_path", P["ptrue"]]

white_box_cmd = [sys.executable, "src/baseline/white_box_baseline.py",
                 "--label-file", P["labels"], "--embedding-file", P["bert"],
                 "--transformed-file", P["transformed"], "--output-file", P["white_box"],
                 "--model-name", WB_MODEL, "--context-file", SOURCE]

if HAS_GPU:
    run(nibs_cmd)
    run(gib_train_cmd)
    ckpt = gib_latest_ckpt()
    if ckpt:
        run([sys.executable, "src/GIBS/evaluation_GIB_wo_mcs.py",
             "--model-path", ckpt, "--embedding-file", P["bert"],
             "--label-file", P["labels"], "--output-file", P["gibs"]])
    run(ptrue_cmd)
    run(white_box_cmd)
else:
    print("No GPU -> run these on a CUDA machine:\n")
    for c in (nibs_cmd, gib_train_cmd, ptrue_cmd, white_box_cmd):
        print("  " + " ".join(c) + "\n")
    print("  # then GIBS inference with the produced checkpoint:")
    print("  python src/GIBS/evaluation_GIB_wo_mcs.py --model-path <ckpt> \\")
    print(f"      --embedding-file {P['bert']} --label-file {P['labels']} --output-file {P['gibs']}")

## 6. Evaluate every method  *(CPU — runs for real)*

We compute **AUROC / AUPRC / Accuracy@80%-coverage** for each method.

- If your demo produced all five method outputs (i.e. you ran Stage 5 on a GPU and
  had real labels), this scores the demo run.
- Otherwise we fall back to the **bundled pre-computed paper results** in
  `data/MorehopQA/Deepseek/` so you always see a real metrics table.

In [ ]:
demo_ready = all(os.path.exists(P[k]) for k in ("labels", "gibs", "nibs", "ptrue", "white_box"))
# A dry-run labels file (all -1) can't produce metrics, so verify there are real labels.
real_labels = False
if os.path.exists(P["labels"]):
    lab = json.load(open(P["labels"]))
    for prob in lab.values():
        for gen in prob.values():
            if any(e["is_correct"] in (0, 1) for e in gen["evaluations"]):
                real_labels = True; break
        if real_labels: break

if demo_ready and real_labels:
    print(f"Scoring the DEMO run ({os.path.dirname(P['labels'])}/).\n")
    run([sys.executable, "src/evaluation/evaluation_metrics.py",
         "--dataset", DATASET, "--model", MODEL])
else:
    print("Demo method outputs / real labels not all present "
          "(expected without a GPU + OpenAI key).")
    print("Falling back to bundled paper results: data/MorehopQA/Deepseek/\n")
    run([sys.executable, "src/evaluation/evaluation_metrics.py",
         "--label-file",     BUNDLED["labels"],
         "--gib-file",       BUNDLED["gibs"],
         "--nibs-file",      BUNDLED["nibs"],
         "--ptrue-file",     BUNDLED["ptrue"],
         "--white-box-file", BUNDLED["white_box"]])

## Recap

You just ran the full path: **generate → transform → bert → label → methods →
evaluate**, all with relative paths writing to `output/`.

To reproduce real results end-to-end, run this notebook (or the equivalent
commands in the [README](../README.md)) on a CUDA machine with an
`OPENAI_API_KEY` set, optionally pointing `SOURCE` at `data/source/morehopqa_full.json`
and raising `NUM_RESPONSES`.

---

## 7. Reproduce the paper results  *(CPU — no GPU / API key needed)*

All datasets and per-model method outputs from the paper are bundled under
`data/<dataset>/<model>/`. This section calls the **same**
[`src/evaluation/evaluation_metrics.py`](../src/evaluation/evaluation_metrics.py)
on each configuration and prints its **AUROC / AUPRC / Accuracy@80%-coverage**
summary table.

- Datasets: `MorehopQA`, `GSM8K`, `Math`  ×  Models: `Llama3.1`, `Phi4`, `Deepseek`.
- Edit the `REPRODUCE` list below to output a specific *(dataset, model)* — e.g.
  `[("MorehopQA", "Deepseek")]` — or keep `[("all", "all")]` for the full grid.
- `GIB-based` is our method (**GIBS**); the other rows are baselines. Higher
  AUROC / AUPRC is better.

(The bundled files have slightly different names across folders, e.g.
`NIBS_results.json` vs `similarity_results.json`; the helper below resolves them
automatically.)

In [ ]:
import os, sys, glob, subprocess

# Make this cell self-contained: locate the repo root (the dir that has both
# data/ and src/) so it works even if the Setup cell above wasn't run first.
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "data")) and os.path.isdir(os.path.join(_d, "src")):
        break
    _d = os.path.dirname(_d)
os.chdir(_d)
print("Repo root:", os.getcwd())

DATA_ROOT = "data"
# Folder names exactly as they appear under data/<dataset>/<model>/
PAPER_DATASETS = ["MorehopQA", "GSM8K", "Math"]
PAPER_MODELS   = ["Llama3.1", "Phi4", "Deepseek"]

# ---- Choose what to output. ("all", "all") runs every dataset x model. ----
# Examples:
#   REPRODUCE = [("MorehopQA", "Deepseek")]              # one config
#   REPRODUCE = [("GSM8K", "all"), ("Math", "Phi4")]     # mix
REPRODUCE = [("all", "all")]

def _find(d, *names):
    """Return the first existing file in d matching one of the given names/globs."""
    for n in names:
        p = os.path.join(d, n)
        if os.path.exists(p):
            return p
        hits = sorted(glob.glob(os.path.join(d, n)))
        if hits:
            return hits[0]
    return None

def discover_paper_files(d):
    # File names differ slightly across folders, so we try a few aliases each.
    return dict(
        label = _find(d, "evaluation.json", "evaluation_results.json"),
        gib   = _find(d, "GIBS_results.json", "GIB_results.json"),
        nibs  = _find(d, "NIBS_results.json", "similarity_results.json", "NIBS_results"),
        ptrue = _find(d, "ptrue_*.pkl"),
        white = _find(d, "white_box_baseline_*.json"),
    )

def reproduce(dataset, model):
    d = os.path.join(DATA_ROOT, dataset, model)
    if not os.path.isdir(d):
        print(f"  (no folder {d}) — skip"); return
    f = discover_paper_files(d)
    missing = [k for k, v in f.items() if v is None]
    if missing:
        print(f"  SKIP {dataset}/{model} — missing files: {missing}"); return
    res = subprocess.run(
        [sys.executable, "src/evaluation/evaluation_metrics.py",
         "--label-file", f["label"], "--gib-file", f["gib"], "--nibs-file", f["nibs"],
         "--ptrue-file", f["ptrue"], "--white-box-file", f["white"]],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = res.stdout.decode("utf-8", "replace")
    # Show only the final summary table (AUROC / AUPRC / Acc@80% per method).
    print(out[out.index("Summary:"):].rstrip() if "Summary:" in out else out.strip()[-800:])

# Expand ("all", ...) selections, then run.
todo = []
for ds, m in REPRODUCE:
    dss = PAPER_DATASETS if ds == "all" else [ds]
    mss = PAPER_MODELS  if m  == "all" else [m]
    for a in dss:
        for b in mss:
            todo.append((a, b))

for ds, m in todo:
    print("=" * 72)
    print(f"### {ds}  /  {m}")
    print("=" * 72)
    reproduce(ds, m)
    print()